In [0]:
# MVP - Engenharia de Dados: Análise de Ativos B3
# Objetivo:** Construir um pipeline Lakehouse (Medallion Architecture) para processamento, modelagem em Star Schema e análise de performance dos ativos da Bovespa.

## Perguntas a responder:
# 1. Como o índice IBOVESPA se comportou ao longo do tempo?
# 2. Existem períodos de maior volatilidade claramente identificáveis?
# 3. Qual a distribuição dos retornos diários do índice?
# 4. Há padrões sazonais (mensais ou anuais) nos retornos?
# 5. A qualidade dos dados é suficiente para análises confiáveis?
# 6. Quais os 20 ativos com maior volume e os 20 com maior volatilidade?
# Metodologia de Organização
# Para responder às perguntas de negócio, o pipeline realiza a **agregação (soma e média)** dos dados por:
# 1. **Ativo (Symbol)**
# 2. **Período (Mês/Ano)**

# Isso permite identificar a liquidez total (Soma do Volume) e a variação de preço (Média de Fechamento) em diferentes janelas temporais.

from pyspark.sql.functions import *
from pyspark.sql.window import Window

source_path = "/Volumes/workspace/default/bovespa_stocks/bovespa_stocks.csv"

# Ingestão com tratamento de separador comum em arquivos Excel/CSV brasileiros
df_raw = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(source_path)

# Se o separador for ponto e vírgula, re-lê o arquivo
if len(df_raw.columns) <= 1:
    df_raw = spark.read.format("csv").option("header", "true").option("inferSchema", "true").option("sep", ";").load(source_path)

# Normalização de nomes de colunas
df_bronze = df_raw.toDF(*[c.lower().replace(" ", "_") for c in df_raw.columns])
df_bronze.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("default.b3_bronze")

df_silver = spark.table("default.b3_bronze")

# 1. Tratamento de Data (Suporta ddMMyyyy e dd/MM/yyyy)
df_silver = df_silver.withColumn("trade_date", coalesce(to_date(col("date"), "ddMMyyyy"), to_date(col("date"), "dd/MM/yyyy")))

# 2. Conversão Numérica
cols_num = ["adj_close", "close", "high", "low", "open", "volume"]
for c in cols_num:
    df_silver = df_silver.withColumn(c, col(c).cast("double"))

# 3. Métricas Financeiras Diárias
w = Window.partitionBy("symbol").orderBy("trade_date")
df_silver = df_silver.withColumn("daily_return", (col("close") - lag(col("close")).over(w)) / lag(col("close")).over(w)) \
                     .withColumn("volatility_21d", stddev(col("daily_return")).over(w.rowsBetween(-20, 0)))

# 4. Qualidade de Dados (DQ)
df_silver = df_silver.withColumn("dq_row_ok", col("trade_date").isNotNull() & col("close").isNotNull())
df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("default.b3_silver")

# Filtramos apenas os dados válidos
silver_valid = spark.table("default.b3_silver").filter("dq_row_ok = true")

# Criando a Tabela Fato Agregada (Soma por Ativo e Período)
# Aqui atendemos ao requisito de organizar somando valores por período
fact_monthly = silver_valid.groupBy("symbol", year("trade_date").alias("ano"), month("trade_date").alias("mes")) \
    .agg(
        sum("volume").alias("total_volume"),           # Soma do volume no período
        avg("close").alias("avg_close"),               # Média de preço no período
        avg("volatility_21d").alias("avg_volatility"), # Média de risco no período
        count("*").alias("dias_negociados")
    )

fact_monthly.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("default.fact_monthly_quotes")

# Tabela Fato Diária (Para detalhes)
silver_valid.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("default.fact_daily_quotes")

print("Modelagem Gold concluída com agregações por período.")

# %sql
# Q1 e Q2: Comportamento do Mercado (Utilizando o Ativo Principal do seu arquivo)
# O gráfico mostrará a média de preço e o volume acumulado por mês
query = """
SELECT 
    ano, mes, 
    AVG(avg_close) as Preco_Medio, 
    SUM(total_volume) as Liquidez_Total
FROM default.fact_monthly_quotes
GROUP BY 1, 2 ORDER BY 1, 2;
"""
df_market = spark.sql(query)
display(df_market)

# %sql
# Q6: 20 ativos com maior VOLUME TOTAL (Soma por Ativo)
query_volume = """
SELECT 
    symbol as Ativo, 
    SUM(total_volume) as Volume_Acumulado
FROM default.fact_monthly_quotes
GROUP BY 1 
ORDER BY 2 DESC 
LIMIT 20
"""
df_volume = spark.sql(query_volume)
display(df_volume)

# Q6: 20 ativos com maior VOLATILIDADE MÉDIA
query_volatility = """
SELECT 
    symbol as Ativo, 
    AVG(avg_volatility) as Risco_Medio
FROM default.fact_monthly_quotes
GROUP BY 1 
ORDER BY 2 DESC 
LIMIT 20
"""
df_volatility = spark.sql(query_volatility)
display(df_volatility)






Modelagem Gold concluída com agregações por período.


ano,mes,Preco_Medio,Liquidez_Total
2010,1,200.90082101217558,3.543699579E9
2010,2,203.60550433297175,3.416138944E9
2010,3,217.557246589563,3.944650254E9
2010,4,220.94794820854963,3.538054911E9
2010,5,199.69150672084965,4.309014384E9
2010,6,212.46217634254964,3.176455681E9
2010,7,213.25439919008966,3.688911474E9
2010,8,219.41790584051913,3.735801834E9
2010,9,229.59759015652594,4.144343868E9
2010,10,226.04321485112337,4.518917802E9


Ativo,Volume_Acumulado
I4,2.2224307606E11
PETR4,1.69834489343E11
MGLU3,1.58499181958E11
BBDC4,9.1339733467E10
ITUB4,8.2100642963E10
COGN3,6.1386501141E10
VIIA3,5.5116603572E10
PETR3,4.4456029179E10
VALE5,3.6528478058E10
CMIG4,3.5951341379E10


Databricks visualization. Run in Databricks to view.

Ativo,Risco_Medio
OSXB3,407.5917375229744
BFPI39,0.33998249561924526
CPTR11,0.3092480901086984
IRBR3,0.17821911467883958
VTRU3,0.17796342216685054
NGRD3,0.1724526961282066
SEQL3,0.15060143486497396
AMOB3,0.15035694227386756
AERI3,0.14441020922525633
BFDN39,0.14349671791080337
